# ITEC631 – AI-Driven Network Intrusion Detection System
## Sprint 3: Two-Stage Detection Pipeline

**Student:** Hayoung  
**Unit:** ITEC631 IT Masters Project Part B, Semester 1 2026

Sprint 3 goal is to build a **two-stage hierarchical classifier**:
- **Stage 1** — Binary classifier: is this traffic BENIGN or ATTACK?
- **Stage 2** — Multi-class classifier: which attack category is it? (DoS/DDoS, PortScan, BruteForce, WebAttack, Botnet, Infiltration)

The two stages are then chained into a single pipeline and compared against a flat single-stage multi-class baseline.

**Key question (RQ1):** Does the two-stage approach outperform a single-stage multi-class classifier on the imbalanced CICIDS2017 dataset?

## Setup

In [ ]:
import os, sys, json, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
RANDOM_STATE = 42

print('libraries loaded')

In [ ]:
ON_KAGGLE = os.path.exists('/kaggle/working')

if ON_KAGGLE:
    # Sprint 1 outputs
    PROC_DIR   = '/kaggle/working/processed_data/'
    MODEL_DIR  = '/kaggle/working/models/'
    DATA_DIR   = None
    if not os.path.exists(PROC_DIR + 'cicids2017_cleaned.csv'):
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'cicids2017_cleaned.csv' in files:
                PROC_DIR = root + '/'
                break
    if not os.path.exists(MODEL_DIR + 'random_forest.pkl'):
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'random_forest.pkl' in files:
                MODEL_DIR = root + '/'
                break
    for root, dirs, files in os.walk('/kaggle/input'):
        if any(f.endswith('.csv') and 'WorkingHours' in f for f in files):
            DATA_DIR = root + '/'
            break
    OUT_DIR = '/kaggle/working/'
else:
    PROC_DIR  = 'processed_data/'
    MODEL_DIR = 'models/'
    DATA_DIR  = 'Dataset/MachineLearningCVE/'
    OUT_DIR   = ''

os.makedirs(PROC_DIR,  exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print(f'PROC_DIR  : {PROC_DIR}')
print(f'MODEL_DIR : {MODEL_DIR}')
print(f'DATA_DIR  : {DATA_DIR}')

## 1. Load & Prepare Data

Loading the full cleaned dataset saved by Sprint 1 (`cicids2017_cleaned.csv`).
This file contains all features plus both label columns (`label_binary` and `label_category`).

If the Sprint 1 file is not available, we fall back to re-processing the raw CSV files from scratch — same pipeline as Sprint 1.

In [ ]:
CATEGORY_MAP = {
    'BENIGN'                     : 'BENIGN',
    'DoS Hulk'                   : 'DoS_DDoS',
    'DoS GoldenEye'              : 'DoS_DDoS',
    'DoS slowloris'              : 'DoS_DDoS',
    'DoS Slowhttptest'           : 'DoS_DDoS',
    'Heartbleed'                 : 'DoS_DDoS',
    'DDoS'                       : 'DoS_DDoS',
    'PortScan'                   : 'PortScan',
    'FTP-Patator'                : 'BruteForce',
    'SSH-Patator'                : 'BruteForce',
    'Web Attack  Brute Force'    : 'WebAttack',
    'Web Attack  XSS'            : 'WebAttack',
    'Web Attack  Sql Injection'  : 'WebAttack',
    'Bot'                        : 'Botnet',
    'Infiltration'               : 'Infiltration',
}

cleaned_path = PROC_DIR + 'cicids2017_cleaned.csv'

if os.path.exists(cleaned_path):
    print('loading from Sprint 1 saved file...')
    full_df = pd.read_csv(cleaned_path)
    print(f'loaded {len(full_df):,} rows x {full_df.shape[1]} cols')
    feature_cols = [c for c in full_df.columns
                    if c not in ('label_binary', 'label_category', 'label_cat_encoded')]
    X = full_df[feature_cols]
    y_binary   = full_df['label_binary']
    y_category = full_df['label_category']

else:
    print('Sprint 1 file not found — re-processing from raw CSVs...')
    csv_files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith('.csv')])
    frames = []
    for fname in csv_files:
        part = pd.read_csv(DATA_DIR + fname, low_memory=False)
        part['source_file'] = fname
        frames.append(part)
        print(f'  loaded {fname} — {len(part):,} rows')

    df = pd.concat(frames, ignore_index=True)
    df.columns = df.columns.str.strip()
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    df.drop_duplicates(inplace=True)
    df.drop(columns=['source_file'], errors='ignore', inplace=True)
    df.reset_index(drop=True, inplace=True)

    df['label_binary']   = (df['Label'] != 'BENIGN').astype(int)
    df['label_category'] = df['Label'].map(CATEGORY_MAP).fillna('Other')

    feature_cols = [c for c in df.columns
                    if c not in ('Label', 'label_binary', 'label_category')]
    X_raw = df[feature_cols]

    # drop constant & duplicate columns
    const_cols = X_raw.columns[X_raw.var() == 0].tolist()
    X_raw.drop(columns=const_cols, inplace=True)
    X_raw = X_raw.loc[:, ~X_raw.columns.duplicated()]

    scaler = MinMaxScaler()
    X = pd.DataFrame(scaler.fit_transform(X_raw), columns=X_raw.columns)
    y_binary   = df['label_binary'].reset_index(drop=True)
    y_category = df['label_category'].reset_index(drop=True)

    le_save = LabelEncoder()
    y_cat_enc = le_save.fit_transform(y_category)
    full_df = X.copy()
    full_df['label_binary']      = y_binary.values
    full_df['label_category']    = y_category.values
    full_df['label_cat_encoded'] = y_cat_enc
    full_df.to_csv(cleaned_path, index=False)
    feature_cols = list(X.columns)
    print(f'processed and saved — {len(full_df):,} rows')

print(f'\nfeatures : {X.shape[1]}')
print(f'\nclass distribution:')
print(y_category.value_counts().to_string())

## 2. Train / Test Split

Using the **same split as Sprint 1** (stratified 80/20, random_state=42) so results are directly comparable.  
We keep the original (pre-SMOTE) labels here — SMOTE is applied separately for each stage.

In [ ]:
X_train_raw, X_test, y_bin_train, y_bin_test = train_test_split(
    X, y_binary,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_binary
)

# get matching category labels using the same indices
y_cat_train = y_category.loc[y_bin_train.index]
y_cat_test  = y_category.loc[y_bin_test.index]

print(f'train  : {len(X_train_raw):,} rows')
print(f'test   : {len(X_test):,} rows')
print(f'\ntrain binary distribution:')
print(y_bin_train.value_counts().rename({0: 'BENIGN', 1: 'ATTACK'}))
print(f'\ntest binary distribution:')
print(y_bin_test.value_counts().rename({0: 'BENIGN', 1: 'ATTACK'}))
print(f'\ntrain category distribution:')
print(y_cat_train.value_counts().to_string())

## 3. Stage 1 — Binary Classifier (BENIGN vs ATTACK)

Loading the best model trained in Sprint 2 as Stage 1.  
If Sprint 2 models are not available, we re-train using the same setup.

In [ ]:
best_model_path = MODEL_DIR + 'best_model.json'
rf_path         = MODEL_DIR + 'random_forest.pkl'

# try to load Sprint 2 best model
stage1_model = None
stage1_name  = None

if os.path.exists(best_model_path):
    with open(best_model_path) as f:
        best_info = json.load(f)
    stage1_name = best_info['best_model']
    model_file  = stage1_name.lower().replace(' ', '_') + '.pkl'
    model_path  = MODEL_DIR + model_file
    if os.path.exists(model_path):
        with open(model_path, 'rb') as f:
            stage1_model = pickle.load(f)
        print(f'loaded Stage 1 from Sprint 2: {stage1_name}  (F1={best_info["f1"]:.4f})')

if stage1_model is None:
    print('Sprint 2 models not found — training Stage 1 from scratch...')
    # apply SMOTE on binary labels
    smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
    X_s1_bal, y_s1_bal = smote.fit_resample(X_train_raw, y_bin_train)
    stage1_model = RandomForestClassifier(
        n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
    )
    stage1_model.fit(X_s1_bal, y_s1_bal)
    stage1_name = 'Random Forest'
    print('Stage 1 training done')

# quick sanity check on test set
s1_pred = stage1_model.predict(X_test)
s1_f1   = f1_score(y_bin_test, s1_pred, average='weighted')
s1_rec  = recall_score(y_bin_test, s1_pred, average='weighted')
print(f'\nStage 1 sanity check on test set:')
print(f'  weighted F1     : {s1_f1:.4f}')
print(f'  weighted recall : {s1_rec:.4f}')
print(classification_report(y_bin_test, s1_pred, target_names=['BENIGN', 'ATTACK']))

## 4. Stage 2 — Attack Category Classifier

Stage 2 only sees records that Stage 1 flagged as ATTACK.  
We train it **only on attack-type records** (BENIGN excluded from training set).  
SMOTE is applied again to handle the imbalance between rare attack categories (e.g. Infiltration, Botnet).

In [ ]:
# filter attack-only records from training set
attack_mask = y_bin_train == 1
X_attack    = X_train_raw[attack_mask]
y_attack    = y_cat_train[attack_mask]

print(f'attack records in training set: {len(X_attack):,}')
print(f'\ncategory distribution (before SMOTE):')
print(y_attack.value_counts().to_string())

# encode category labels
le2 = LabelEncoder()
y_attack_enc = le2.fit_transform(y_attack)

print(f'\ncategory encoding:')
for cls, idx in zip(le2.classes_, range(len(le2.classes_))):
    print(f'  {cls:<16} → {idx}')

In [ ]:
# apply SMOTE to balance attack categories
# need at least k_neighbors+1 samples per class
min_class_count = pd.Series(y_attack_enc).value_counts().min()
k = min(5, min_class_count - 1)
print(f'min class count: {min_class_count}  →  using k_neighbors={k}')

smote2 = SMOTE(random_state=RANDOM_STATE, k_neighbors=k)
X_atk_bal, y_atk_bal = smote2.fit_resample(X_attack, y_attack_enc)

print(f'\nSMOTE result:')
print(f'  before: {len(X_attack):,} rows')
print(f'  after : {len(X_atk_bal):,} rows')
print(f'\ncategory distribution after SMOTE:')
bal_series = pd.Series(le2.inverse_transform(y_atk_bal))
print(bal_series.value_counts().to_string())

### 4.1 Train Three Classifiers for Stage 2

In [ ]:
print('training Stage 2 classifiers...')

s2_rf = RandomForestClassifier(
    n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
)
s2_rf.fit(X_atk_bal, y_atk_bal)
print('  Random Forest — done')

s2_xgb = XGBClassifier(
    n_estimators=100, random_state=RANDOM_STATE,
    num_class=len(le2.classes_)
)
s2_xgb.fit(X_atk_bal, y_atk_bal)
print('  XGBoost       — done')

s2_dt = DecisionTreeClassifier(max_depth=20, random_state=RANDOM_STATE)
s2_dt.fit(X_atk_bal, y_atk_bal)
print('  Decision Tree — done')

print('\nall Stage 2 classifiers trained')

### 4.2 Evaluate Stage 2 in Isolation

Evaluating Stage 2 on true attack records only from the test set. This tells us how well each classifier identifies attack categories — independent of Stage 1 errors.

In [ ]:
# ground truth: test records that are actually attacks
attack_test_mask  = y_bin_test == 1
X_test_attacks    = X_test[attack_test_mask]
y_test_cat_true   = y_cat_test[attack_test_mask]
y_test_cat_enc    = le2.transform(
    [c if c in le2.classes_ else le2.classes_[0]
     for c in y_test_cat_true]
)

def eval_stage2(name, model):
    y_pred_enc  = model.predict(X_test_attacks)
    y_pred_cat  = le2.inverse_transform(y_pred_enc)
    f1  = f1_score(y_test_cat_enc, y_pred_enc, average='weighted')
    rec = recall_score(y_test_cat_enc, y_pred_enc, average='weighted')
    acc = accuracy_score(y_test_cat_enc, y_pred_enc)
    print(f'\n{name}  (Stage 2 isolation)')
    print('-' * 45)
    print(f'  accuracy  : {acc:.4f}')
    print(f'  recall    : {rec:.4f}')
    print(f'  f1 score  : {f1:.4f}')
    print()
    print(classification_report(
        y_test_cat_enc, y_pred_enc,
        target_names=le2.classes_,
        zero_division=0
    ))
    return {'name': name, 'model': model,
            'acc': acc, 'rec': rec, 'f1': f1,
            'y_pred_enc': y_pred_enc}

s2_results = []
s2_results.append(eval_stage2('Random Forest', s2_rf))
s2_results.append(eval_stage2('XGBoost',       s2_xgb))
s2_results.append(eval_stage2('Decision Tree', s2_dt))

best_s2 = max(s2_results, key=lambda r: r['f1'])
print(f'best Stage 2 model: {best_s2["name"]}  (F1={best_s2["f1"]:.4f})')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Stage 2 Confusion Matrices — Attack Category Classification',
             fontsize=13, fontweight='bold')

for ax, res in zip(axes, s2_results):
    cm  = confusion_matrix(y_test_cat_enc, res['y_pred_enc'])
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Oranges', ax=ax,
                xticklabels=le2.classes_,
                yticklabels=le2.classes_,
                cbar=False)
    ax.set_title(f'{res["name"]}\nF1={res["f1"]:.4f}',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.tick_params(axis='x', rotation=30)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig(OUT_DIR + 'stage2_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved stage2_confusion.png')

## 5. Two-Stage Pipeline

Chain Stage 1 and Stage 2 into a single `TwoStageClassifier`.

Prediction logic:
1. Stage 1 classifies every record as BENIGN (0) or ATTACK (1)
2. Records predicted as ATTACK are passed to Stage 2 for category classification
3. Records predicted as BENIGN get the label `'BENIGN'`

This mirrors how a real IDS would work: only flag suspected attacks for deeper analysis.

In [ ]:
class TwoStageClassifier:
    """
    Chains a binary Stage 1 classifier with a multi-class Stage 2 classifier.

    Stage 1 : binary  — BENIGN vs ATTACK
    Stage 2 : multi   — attack category (only called for Stage 1 ATTACKs)
    """

    def __init__(self, stage1, stage2, label_encoder):
        self.stage1 = stage1
        self.stage2 = stage2
        self.le     = label_encoder   # fitted LabelEncoder for Stage 2

    def predict(self, X):
        X = np.array(X) if not isinstance(X, np.ndarray) else X

        # Stage 1 — binary
        s1_pred   = self.stage1.predict(X)
        final_pred = np.array(['BENIGN'] * len(X), dtype=object)

        # Stage 2 — only for predicted attacks
        attack_idx = np.where(s1_pred == 1)[0]
        if len(attack_idx) > 0:
            X_attacks       = X[attack_idx]
            s2_pred_enc     = self.stage2.predict(X_attacks)
            s2_pred_labels  = self.le.inverse_transform(s2_pred_enc)
            final_pred[attack_idx] = s2_pred_labels

        return final_pred

    def predict_proba_stage1(self, X):
        return self.stage1.predict_proba(X)

print('TwoStageClassifier defined')

In [ ]:
# build pipeline using best Stage 2 model
pipeline = TwoStageClassifier(
    stage1         = stage1_model,
    stage2         = best_s2['model'],
    label_encoder  = le2
)

# full 7-class evaluation on test set
# ground truth: map y_cat_test (which includes 'BENIGN') as-is
y_pipeline_pred = pipeline.predict(np.array(X_test))

# align classes: BENIGN + attack categories
all_classes = ['BENIGN'] + list(le2.classes_)

# encode ground truth and predictions consistently
le_full = LabelEncoder()
le_full.fit(all_classes)
y_true_enc = le_full.transform(y_cat_test.values)
y_pipe_enc = le_full.transform(y_pipeline_pred)

pipe_f1  = f1_score(y_true_enc, y_pipe_enc, average='weighted')
pipe_rec = recall_score(y_true_enc, y_pipe_enc, average='weighted')
pipe_acc = accuracy_score(y_true_enc, y_pipe_enc)

print('Two-Stage Pipeline — Full Test Set Evaluation')
print('=' * 55)
print(f'  accuracy         : {pipe_acc:.4f}')
print(f'  weighted recall  : {pipe_rec:.4f}')
print(f'  weighted F1      : {pipe_f1:.4f}')
print()
print(classification_report(
    y_true_enc, y_pipe_enc,
    target_names=all_classes,
    zero_division=0
))

## 6. Single-Stage Baseline

Training a **flat multi-class classifier** (same Random Forest settings) that predicts all 7 classes in one shot — BENIGN plus all 6 attack categories.  
This is the baseline we compare against to answer **RQ1**.

In [ ]:
# encode all 7 categories
le_full_train = LabelEncoder()
le_full_train.fit(all_classes)

y_train_full = y_cat_train.values
y_test_full  = y_cat_test.values

# apply SMOTE on full 7-class labels
y_train_full_enc = le_full_train.transform(y_train_full)

min_c = pd.Series(y_train_full_enc).value_counts().min()
k_full = min(5, min_c - 1)

smote_full = SMOTE(random_state=RANDOM_STATE, k_neighbors=k_full)
X_full_bal, y_full_bal = smote_full.fit_resample(X_train_raw, y_train_full_enc)

print(f'SMOTE 7-class:  {len(X_train_raw):,} → {len(X_full_bal):,} rows')
print()

# train single-stage RF
print('training single-stage Random Forest...')
single_rf = RandomForestClassifier(
    n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
)
single_rf.fit(X_full_bal, y_full_bal)
print('done')

# evaluate
y_single_pred = single_rf.predict(np.array(X_test))
y_test_full_enc = le_full_train.transform(y_test_full)

single_f1  = f1_score(y_test_full_enc, y_single_pred, average='weighted')
single_rec = recall_score(y_test_full_enc, y_single_pred, average='weighted')
single_acc = accuracy_score(y_test_full_enc, y_single_pred)

print()
print('Single-Stage Classifier — Full Test Set Evaluation')
print('=' * 55)
print(f'  accuracy         : {single_acc:.4f}')
print(f'  weighted recall  : {single_rec:.4f}')
print(f'  weighted F1      : {single_f1:.4f}')
print()
print(classification_report(
    y_test_full_enc, y_single_pred,
    target_names=le_full_train.classes_,
    zero_division=0
))

## 7. Head-to-Head Comparison: Single-Stage vs Two-Stage

In [ ]:
# --- summary table ---
comparison = pd.DataFrame([
    {
        'Approach'   : 'Single-Stage (RF)',
        'Accuracy'   : f'{single_acc:.4f}',
        'Recall'     : f'{single_rec:.4f}',
        'Weighted F1': f'{single_f1:.4f}',
    },
    {
        'Approach'   : f'Two-Stage ({stage1_name} + {best_s2["name"]})',
        'Accuracy'   : f'{pipe_acc:.4f}',
        'Recall'     : f'{pipe_rec:.4f}',
        'Weighted F1': f'{pipe_f1:.4f}',
    },
])
print(comparison.to_string(index=False))

delta_f1  = pipe_f1  - single_f1
delta_rec = pipe_rec - single_rec
delta_acc = pipe_acc - single_acc

print()
print(f'Δ F1       : {delta_f1:+.4f}  ({"two-stage better" if delta_f1 > 0 else "single-stage better"})')
print(f'Δ Recall   : {delta_rec:+.4f}  ({"two-stage better" if delta_rec > 0 else "single-stage better"})')
print(f'Δ Accuracy : {delta_acc:+.4f}  ({"two-stage better" if delta_acc > 0 else "single-stage better"})')

In [ ]:
# --- bar chart ---
metrics   = ['Accuracy', 'Recall', 'Weighted F1']
single_scores = [single_acc, single_rec, single_f1]
pipe_scores   = [pipe_acc,   pipe_rec,   pipe_f1]

x     = np.arange(len(metrics))
width = 0.35
colors = ['#3498db', '#e74c3c']

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - width/2, single_scores, width, label='Single-Stage RF',
            color=colors[0], edgecolor='white', alpha=0.9)
b2 = ax.bar(x + width/2, pipe_scores, width,
            label=f'Two-Stage ({stage1_name} + {best_s2["name"]})',
            color=colors[1], edgecolor='white', alpha=0.9)

for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.002,
                f'{bar.get_height():.4f}',
                ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(0.80, 1.02)
ax.set_ylabel('Score')
ax.set_title('Single-Stage vs Two-Stage Pipeline — Full 7-Class Evaluation',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))

plt.tight_layout()
plt.savefig(OUT_DIR + 'pipeline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved pipeline_comparison.png')

### 7.1 Per-Class Recall: Where Does Each Approach Win?

Recall is the most important metric for an IDS — a missed attack is worse than a false positive.

In [ ]:
from sklearn.metrics import recall_score

classes = all_classes

# single-stage per-class recall
single_recalls = {}
for i, cls in enumerate(le_full_train.classes_):
    mask = y_test_full_enc == i
    if mask.sum() > 0:
        single_recalls[cls] = recall_score(
            y_test_full_enc == i,
            y_single_pred   == i
        )

# two-stage per-class recall
pipe_recalls = {}
for cls in all_classes:
    true_mask = y_cat_test.values == cls
    pred_mask = y_pipeline_pred  == cls
    if true_mask.sum() > 0:
        pipe_recalls[cls] = recall_score(true_mask, pred_mask)

# build comparison dataframe
class_recall_df = pd.DataFrame({
    'Class'         : classes,
    'Single-Stage'  : [single_recalls.get(c, 0) for c in classes],
    'Two-Stage'     : [pipe_recalls.get(c, 0)   for c in classes],
})
class_recall_df['Δ Recall'] = (class_recall_df['Two-Stage']
                                - class_recall_df['Single-Stage'])
class_recall_df = class_recall_df.sort_values('Δ Recall', ascending=False)

print('Per-Class Recall Comparison:')
print(class_recall_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

# plot
fig, ax = plt.subplots(figsize=(11, 5))
x   = np.arange(len(class_recall_df))
w   = 0.35
b1  = ax.bar(x - w/2, class_recall_df['Single-Stage'],  w,
             label='Single-Stage',  color='#3498db', edgecolor='white', alpha=0.9)
b2  = ax.bar(x + w/2, class_recall_df['Two-Stage'],   w,
             label='Two-Stage',     color='#e74c3c', edgecolor='white', alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels(class_recall_df['Class'], rotation=20, ha='right')
ax.set_ylim(0, 1.10)
ax.set_ylabel('Recall')
ax.set_title('Per-Class Recall — Single-Stage vs Two-Stage', fontsize=12, fontweight='bold')
ax.axhline(1.0, color='gray', linestyle='--', linewidth=0.8)
ax.legend(fontsize=10)

for bars in [b1, b2]:
    for bar in bars:
        if bar.get_height() > 0.01:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.01,
                    f'{bar.get_height():.3f}',
                    ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig(OUT_DIR + 'per_class_recall.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved per_class_recall.png')

## 8. Error Analysis

Understanding where the two-stage pipeline makes mistakes:
1. **Stage 1 False Negatives** — attacks that slip through as BENIGN (most dangerous)
2. **Stage 2 Misclassifications** — attacks detected but wrong category assigned

In [ ]:
# Stage 1 errors
s1_pred_test  = stage1_model.predict(np.array(X_test))
fn_mask = (y_bin_test.values == 1) & (s1_pred_test == 0)   # attack missed
fp_mask = (y_bin_test.values == 0) & (s1_pred_test == 1)   # benign flagged

print('Stage 1 Error Breakdown:')
print(f'  False Negatives (missed attacks) : {fn_mask.sum():,}  '
      f'({fn_mask.sum()/y_bin_test.sum()*100:.2f}% of all attacks)')
print(f'  False Positives (benign flagged) : {fp_mask.sum():,}  '
      f'({fp_mask.sum()/(y_bin_test==0).sum()*100:.2f}% of all benign)')

# which attack types does Stage 1 miss most?
if fn_mask.sum() > 0:
    fn_categories = y_cat_test.values[fn_mask]
    fn_series     = pd.Series(fn_categories).value_counts()
    print(f'\nAttack types most missed by Stage 1:')
    print(fn_series.to_string())

print()
# Stage 2 errors (on true attack records that Stage 1 correctly flagged)
correctly_flagged = (y_bin_test.values == 1) & (s1_pred_test == 1)
X_cf  = np.array(X_test)[correctly_flagged]
y_cf  = y_cat_test.values[correctly_flagged]

if len(X_cf) > 0:
    s2_pred_cf_enc  = best_s2['model'].predict(X_cf)
    s2_pred_cf_cat  = le2.inverse_transform(s2_pred_cf_enc)
    stage2_errors   = (y_cf != s2_pred_cf_cat)

    print(f'Stage 2 Error Breakdown (on correctly flagged attacks):')
    print(f'  Total correctly flagged : {len(X_cf):,}')
    print(f'  Stage 2 misclassified   : {stage2_errors.sum():,}  '
          f'({stage2_errors.sum()/len(X_cf)*100:.2f}%)')

    # which categories get confused?
    if stage2_errors.sum() > 0:
        confused_true = y_cf[stage2_errors]
        confused_pred = s2_pred_cf_cat[stage2_errors]
        confusion_pairs = pd.Series(
            [f'{t} → {p}' for t, p in zip(confused_true, confused_pred)]
        ).value_counts().head(10)
        print(f'\nTop confusion pairs (true → predicted):')
        print(confusion_pairs.to_string())

## 9. Save Stage 2 Models

In [ ]:
os.makedirs(MODEL_DIR, exist_ok=True)

stage2_map = {
    'stage2_random_forest'  : s2_rf,
    'stage2_xgboost'        : s2_xgb,
    'stage2_decision_tree'  : s2_dt,
}
for name, model in stage2_map.items():
    path = MODEL_DIR + name + '.pkl'
    with open(path, 'wb') as f:
        pickle.dump(model, f)
    print(f'saved {path}')

# save label encoder for Stage 2
with open(MODEL_DIR + 'stage2_label_encoder.pkl', 'wb') as f:
    pickle.dump(le2, f)
print(f'saved {MODEL_DIR}stage2_label_encoder.pkl')

# save full-class label encoder
with open(MODEL_DIR + 'full_label_encoder.pkl', 'wb') as f:
    pickle.dump(le_full, f)
print(f'saved {MODEL_DIR}full_label_encoder.pkl')

# save Sprint 3 results summary
sprint3_summary = {
    'stage1_model'       : stage1_name,
    'stage2_model'       : best_s2['name'],
    'single_stage_f1'    : round(single_f1,  4),
    'single_stage_rec'   : round(single_rec, 4),
    'single_stage_acc'   : round(single_acc, 4),
    'pipeline_f1'        : round(pipe_f1,    4),
    'pipeline_rec'       : round(pipe_rec,   4),
    'pipeline_acc'       : round(pipe_acc,   4),
    'delta_f1'           : round(delta_f1,   4),
    'stage2_classes'     : list(le2.classes_),
    'all_classes'        : all_classes,
}
with open(MODEL_DIR + 'sprint3_results.json', 'w') as f:
    json.dump(sprint3_summary, f, indent=2)
print(f'saved {MODEL_DIR}sprint3_results.json')

## 10. Sprint 3 Summary

In [ ]:
print('Sprint 3 — Two-Stage Pipeline Complete')
print('=' * 60)
print()
print(f'Stage 1 : {stage1_name:<20}  (BENIGN vs ATTACK)')
print(f'Stage 2 : {best_s2["name"]:<20}  (attack category)')
print()
print('Performance Summary:')
print(f'  {"Approach":<40} {"Accuracy":>9}  {"Recall":>9}  {"F1":>9}')
print(f'  {"-"*40}  {"-"*9}  {"-"*9}  {"-"*9}')
print(f'  {"Single-Stage Random Forest":<40} {single_acc:>9.4f}  {single_rec:>9.4f}  {single_f1:>9.4f}')
_two_stage_label = f"Two-Stage ({stage1_name} + {best_s2['name']})"
print(f'  {_two_stage_label:<40} {pipe_acc:>9.4f}  {pipe_rec:>9.4f}  {pipe_f1:>9.4f}')
print()
print(f'  Δ F1 = {delta_f1:+.4f} in favour of the {"two-stage" if delta_f1 > 0 else "single-stage"} approach')
print()
print('Saved outputs:')
print(f'  {MODEL_DIR}stage2_random_forest.pkl')
print(f'  {MODEL_DIR}stage2_xgboost.pkl')
print(f'  {MODEL_DIR}stage2_decision_tree.pkl')
print(f'  {MODEL_DIR}stage2_label_encoder.pkl')
print(f'  {MODEL_DIR}sprint3_results.json')
print()
print('Next: Sprint 4 — SHAP explainability on the two-stage pipeline')
print('      Generate feature-level explanations for every prediction')